# Analytic $\alpha_{\mathrm{rec}}$ Estimation for the 200k V1 Model (Updated for Delays, ASC, Refractory, Multi-Receptor)

This notebook keeps only:
1. The analytic calculation of $\alpha_{\mathrm{rec}}$ for the 200k model.
2. The detailed derivation for this case.

Compared with the previous simplified version, this update explicitly accounts for:
- synaptic **delays** (via delay-aware attenuation in the reduced coupling),
- local dynamics from **ASC** and **refractory gating**,
- full **multi-receptor** PSC basis collapse (not single basis).

Assumption for final estimate: $\gamma=1$ and tune only $\alpha_{\mathrm{rec}}$.

In [39]:
import numpy as np
import pickle as pkl
import os
from scipy import sparse
from scipy.sparse.linalg import eigs

from v1_model_utils.load_sparse import load_network

# -----------------------------
# Configuration
# -----------------------------
DATA_DIR = 'GLIF_network_nll_full'
N_NEURONS = 203816
DT = 1.0
LR_SCALE = 1.0             # should match training flag lr_scale
MAX_DELAY_STEPS = 5        # model default max_delay clipping

# alpha_rec estimation settings (gamma fixed to 1)
GAMMA = 1.0
RATES_HZ = [2, 5, 10, 20]
NEAR_THR_FACTOR = 5.0
KAPPA = 0.7                # safety factor in [0.5, 0.8]

print(f"Config: N={N_NEURONS:,}, dt={DT} ms, gamma={GAMMA}, rates={RATES_HZ}, kappa={KAPPA}")

Config: N=203,816, dt=1.0 ms, gamma=1.0, rates=[2, 5, 10, 20], kappa=0.7


In [40]:
# Load full 200k network
network = load_network(
    data_dir=DATA_DIR,
    core_only=False,
    n_neurons=N_NEURONS,
    seed=3000,
    connected_selection=True,
    tensorflow_speed_up=False,
)

node_params = network['node_params']
node_type_ids = network['node_type_ids']

# Membrane/current factors (match models.py)
V_th = node_params['V_th'][node_type_ids]
E_L = node_params['E_L'][node_type_ids]
C_m = node_params['C_m'][node_type_ids]
g = node_params['g'][node_type_ids]

voltage_scale = V_th - E_L
tau_m = C_m / g
alpha_m = np.exp(-DT / tau_m)                          # membrane decay
current_factor = (1.0 - alpha_m) / g                   # B_i
alpha_m_bar = float(np.mean(alpha_m))

# Refractory stats (for surrogate active-fraction correction)
t_ref_ms = node_params['t_ref'][node_type_ids]
mean_t_ref_ms = float(np.mean(t_ref_ms))

# ASC stats (for local feedback correction)
# models.py normalizes asc_amps by voltage_scale before use
asc_amps_norm = node_params['asc_amps'][node_type_ids] / voltage_scale[:, None]  # [N,2]
k_rates = node_params['k'][node_type_ids]                                       # [N,2]
asc_decay = np.exp(-DT * k_rates)
# one-step closure local ASC feedback coefficient per neuron
asc_feedback_per_neuron = np.sum(current_factor[:, None] * asc_amps_norm / (1.0 - asc_decay), axis=1)
asc_feedback_bar = float(np.mean(asc_feedback_per_neuron))

# Recurrent synapse tables
indices = np.array(network['synapses']['indices'])
post_ids = indices[:, 0]
pre_ids = indices[:, 1]
raw_weights = np.array(network['synapses']['weights'], dtype=np.float64)
syn_ids = np.array(network['synapses']['syn_ids'])
raw_delays = np.array(network['synapses']['delays'], dtype=np.float64)
delay_steps = np.round(np.clip(raw_delays, DT, MAX_DELAY_STEPS) / DT).astype(np.int32)

# Match model recurrent weight scaling by postsynaptic voltage scale
w_scaled = raw_weights / voltage_scale[post_ids]

# Multi-receptor PSC collapse with corrected factors
with open(os.path.join(DATA_DIR, 'tf_data', 'syn_id_to_syn_weights_dict.pkl'), 'rb') as f:
    syn_id_to_syn_weights_dict = pkl.load(f)
synaptic_basis_weights = np.array(list(syn_id_to_syn_weights_dict.values()), dtype=np.float64)  # [n_syn_types, R]

tau_basis = np.load('synaptic_data/tau_basis.npy').astype(np.float64)
d_syn = np.exp(-DT / tau_basis)
psc_initial = np.e / tau_basis

# receptor gain from c->i elimination in one-step closure
# gain_r = (e/tau_r) * (DT * d_r / (1-d_r)^2)
receptor_gain = psc_initial * (DT * d_syn / (1.0 - d_syn) ** 2)

# collapse basis weights across receptors (signed, model-faithful linearization)
collapsed_basis_per_type = synaptic_basis_weights @ receptor_gain  # [n_syn_types]

# Base no-delay effective connection weight per synapse
# delay will be injected later via q^(d-1)
base_conn = (
    current_factor[post_ids] *
    LR_SCALE *
    w_scaled *
    collapsed_basis_per_type[syn_ids]
)

W_base = sparse.csr_matrix((base_conn, (post_ids, pre_ids)), shape=(N_NEURONS, N_NEURONS))

print(f"Network loaded: neurons={network['n_nodes']:,}, synapses={len(base_conn):,}")
print(f"alpha_m_bar = {alpha_m_bar:.6f}")
print(f"mean_t_ref_ms = {mean_t_ref_ms:.3f}")
print(f"asc_feedback_bar = {asc_feedback_bar:.6f}")
print(f"delay-step fractions: "
      f"d=1:{np.mean(delay_steps==1):.3f}, d=2:{np.mean(delay_steps==2):.3f}, d>=3:{np.mean(delay_steps>=3):.3f}")
print(f"W_base nnz = {W_base.nnz:,}")

Loading network_dat.pkl file...
> Maximum sample radius: 700.00
> Number of Neurons: 203816
> Number of Synapses: 84132910
Network loaded: neurons=203,816, synapses=84,132,910
alpha_m_bar = 0.932079
mean_t_ref_ms = 4.055
asc_feedback_bar = -1.692468
delay-step fractions: d=1:0.659, d=2:0.294, d>=3:0.047
W_base nnz = 84,132,910


In [41]:
print("Computing rho(W_base)...")
try:
    eigvals, _ = eigs(W_base.astype(np.float64), k=1, which='LM', tol=1e-3, maxiter=700)
    rhoW_base = float(np.max(np.abs(eigvals)))
except Exception as e:
    one_norm = float(sparse.linalg.norm(W_base, ord=1))
    inf_norm = float(sparse.linalg.norm(W_base, ord=np.inf))
    rhoW_base = float(np.sqrt(one_norm * inf_norm))
    print(f"eigs failed ({e}); using sqrt(||W||_1 ||W||_inf) upper estimate")

print(f"rho(W_base) = {rhoW_base:.6f}")


def sbar_raw_from_rate(rate_hz, dt_ms=1.0, near_thr_factor=5.0):
    p_spike = rate_hz * dt_ms / 1000.0
    p_near = min(near_thr_factor * p_spike, 1.0)
    return 0.5 * p_near


def refractory_open_fraction(rate_hz, mean_t_ref_ms):
    # Mean availability under Poisson-like occupancy approximation
    return max(0.0, 1.0 - rate_hz * mean_t_ref_ms / 1000.0)


def local_rho_gamma1(rate_hz):
    s_raw = sbar_raw_from_rate(rate_hz, DT, NEAR_THR_FACTOR)
    eta_ref = refractory_open_fraction(rate_hz, mean_t_ref_ms)
    s_eff = eta_ref * s_raw
    # Local coefficient from (lambda_m - s) + ASC correction s*asc_feedback
    a_local = alpha_m_bar + s_eff * (asc_feedback_bar - 1.0)
    return s_raw, s_eff, abs(a_local)


def delay_factor_from_q(q, delay_steps, weights_abs):
    # Envelope-based delay attenuation: contribution from delay d weighted by q^(d-1)
    q = float(np.clip(q, 0.0, 0.9999))
    return float(np.average(np.power(q, delay_steps - 1), weights=weights_abs))


def alpha_crit_gamma1(rate_hz):
    s_raw, s_eff, rho_local = local_rho_gamma1(rate_hz)
    if s_eff <= 0:
        return s_raw, s_eff, rho_local, np.nan, np.nan, np.inf

    # Delay-aware recurrent spectral proxy
    d_factor = delay_factor_from_q(rho_local, delay_steps, np.abs(base_conn))
    rhoW_eff = rhoW_base * d_factor

    if rho_local >= 1.0:
        # unstable even at alpha_rec=0 in this proxy
        return s_raw, s_eff, rho_local, d_factor, rhoW_eff, 0.0

    den = s_eff * rhoW_eff
    if den <= 0:
        return s_raw, s_eff, rho_local, d_factor, rhoW_eff, np.inf

    acrit = (1.0 - rho_local) / den
    return s_raw, s_eff, rho_local, d_factor, rhoW_eff, max(acrit, 0.0)


rows = []
for r in RATES_HZ:
    rows.append((r, *alpha_crit_gamma1(r)))

finite = [row[-1] for row in rows if np.isfinite(row[-1]) and row[-1] > 0]
if len(finite) == 0:
    raise RuntimeError("No finite positive alpha_crit found. Check assumptions/inputs.")

alpha_crit_min = float(min(finite))
alpha_rec_opt = float(KAPPA * alpha_crit_min)
alpha_rec_range = (0.5 * alpha_crit_min, 0.8 * alpha_crit_min)

print()
print("=" * 90)
print("FINAL ANALYTIC alpha_rec FOR THIS 200k MODEL (gamma = 1, delay/ASC/refractory aware)")
print("=" * 90)
print(f"alpha_m_bar                      : {alpha_m_bar:.6f}")
print(f"asc_feedback_bar                 : {asc_feedback_bar:.6f}")
print(f"rho(W_base)                      : {rhoW_base:.6f}")
print(f"gamma                            : 1.000")
print(f"rates (Hz)                       : {RATES_HZ}")
print(f"near-threshold factor            : {NEAR_THR_FACTOR:.2f}")
print(f"mean refractory (ms)             : {mean_t_ref_ms:.3f}")
print(f"kappa                            : {KAPPA:.2f}")

print()
print(f"{'rate':<6} {'s_raw':<10} {'s_eff':<10} {'rho_local':<11} {'delay_fac':<10} {'rhoW_eff':<11} {'alpha_crit':<12}")
print("-" * 78)
for r, s_raw, s_eff, rho_local, d_fac, rhoW_eff, acrit in rows:
    print(f"{r:<6} {s_raw:<10.5f} {s_eff:<10.5f} {rho_local:<11.5f} {d_fac:<10.5f} {rhoW_eff:<11.5f} {acrit:<12.6f}")

print()
print("Robust recommendation:")
print(f"alpha_crit_min                   : {alpha_crit_min:.6f}")
print(f"alpha_rec_opt = kappa*min        : {alpha_rec_opt:.6f}")
print(f"practical range (0.5-0.8)*crit   : [{alpha_rec_range[0]:.6f}, {alpha_rec_range[1]:.6f}]")

Computing rho(W_base)...
rho(W_base) = 14.540479

FINAL ANALYTIC alpha_rec FOR THIS 200k MODEL (gamma = 1, delay/ASC/refractory aware)
alpha_m_bar                      : 0.932079
asc_feedback_bar                 : -1.692468
rho(W_base)                      : 14.540479
gamma                            : 1.000
rates (Hz)                       : [2, 5, 10, 20]
near-threshold factor            : 5.00
mean refractory (ms)             : 4.055
kappa                            : 0.70

rate   s_raw      s_eff      rho_local   delay_fac  rhoW_eff    alpha_crit  
------------------------------------------------------------------------------
2      0.00500    0.00496    0.91873     0.97411    14.16407    1.156994    
5      0.01250    0.01225    0.89911     0.96793    14.07411    0.585372    
10     0.02500    0.02399    0.86750     0.95801    13.92991    0.396566    
20     0.05000    0.04595    0.80837     0.93963    13.66264    0.305268    

Robust recommendation:
alpha_crit_min                

## Detailed Derivation for the Model-Case Approximation

### 1) Model-consistent forward state (multi-receptor, delays, ASC)
For neuron index $j$, receptor $r=1,\dots,R$, ASC channel $q=1,2$:

\begin{align}
v_{j,t+1}
&= \alpha_{m,j} v_{j,t}
+ B_{i,j}\Big(\sum_{r=1}^R i_{j,t}^{(r)} + \sum_{q=1}^2 a_{j,t}^{(q)}\Big)
- z_{j,t}, \tag{1}\\
z_{j,t} &= H(v_{j,t}-1), \tag{2}\\
c_{j,t+1}^{(r)} &= d_r c_{j,t}^{(r)} + c_r\,I_{j,t}^{(r)}, \tag{3}\\
i_{j,t+1}^{(r)} &= d_r i_{j,t}^{(r)} + \Delta t\,d_r c_{j,t}^{(r)}, \tag{4}\\
a_{j,t+1}^{(q)} &= \lambda_{a,j}^{(q)} a_{j,t}^{(q)} + A_{j}^{(q)} z_{j,t}. \tag{5}
\end{align}

Recurrent input with delay buckets:
\begin{align}
I_{j,t}^{(r)}=\ell_s\sum_i w_{ji}\,b_r(\text{syn}_{ji})\,\tilde z_{i,t-d_{ji}+1}. \tag{6}
\end{align}

### 2) Surrogate and recurrent dampening

\begin{align}
\Gamma_t := \frac{\partial z_t}{\partial v_t}=\gamma\,\mathrm{diag}(\phi_t), \tag{7}
\end{align}
and recurrent STE
\begin{align}
\tilde z_t=\mathrm{stopgrad}(z_t-\alpha_{\mathrm{rec}}z_t)+\alpha_{\mathrm{rec}}z_t
\Rightarrow
\frac{\partial\tilde z_t}{\partial v_t}=\alpha_{\mathrm{rec}}\Gamma_t. \tag{8}
\end{align}

### 3) Jacobian block giving recurrent coupling
For receptor $r$,
\begin{align}
\frac{\partial c_{t+1}^{(r)}}{\partial v_t}
= c_r\,W_r\,\frac{\partial\tilde z_{t-d+1}}{\partial v_t}
\sim \alpha_{\mathrm{rec}}\,W_r\,\Gamma_t
\end{align}
for delay-1 paths, with delayed paths entering through shift-chain states. This is the origin of the $\alpha_{\mathrm{rec}}\mathcal W\Gamma_t$ block.

### 4) Eliminating synaptic/ASC adjoints
Collect non-voltage adjoints into
\begin{align}
x_t=[\bar c_t^{(1:R)},\bar i_t^{(1:R)},\bar a_t^{(1:2)}]^\top.
\end{align}
Then linearized backward recursion has form
\begin{align}
x_t = F x_{t+1} + G\bar v_{t+1}, \tag{9}
\end{align}
with $\rho(F)<1$ since all decays are in $(0,1)$.
Unrolling:
\begin{align}
x_{t+1}=\sum_{k\ge0}F^kG\,\bar v_{t+2+k}. \tag{10}
\end{align}

To obtain a tractable one-step closure, assume envelope
\begin{align}
\bar v_{t+2+k}\approx q^k\bar v_{t+1},\quad 0<q<1. \tag{11}
\end{align}
Then
\begin{align}
x_{t+1}\approx (I-qF)^{-1}G\,\bar v_{t+1}. \tag{12}
\end{align}
This yields effective maps:
- $K_c(q)$ from $\bar v_{t+1}$ to receptor-rise adjoints,
- $K_a(q)$ from $\bar v_{t+1}$ to ASC adjoints.

Substitution into voltage adjoint gives
\begin{align}
\bar v_t\approx \Big(A_t + \alpha_{\mathrm{rec}}W_{\mathrm{eff}}(q)\Gamma_t\Big)^\top\bar v_{t+1}, \tag{13}
\end{align}
with
\begin{align}
A_t &\approx \mathrm{diag}(\alpha_m) - \Gamma_t + \Gamma_t\,\mathrm{diag}(\kappa_{\mathrm{asc}}), \\
\kappa_{\mathrm{asc},j} &\approx B_{i,j}\sum_q\frac{A_j^{(q)}}{1-\lambda_{a,j}^{(q)}}. \tag{14}
\end{align}

### 5) Multi-receptor effective coupling with delay attenuation
For one-step closure of receptor dynamics,
\begin{align}
\text{gain}_r = \frac{e}{\tau_r}\cdot\frac{\Delta t\,d_r}{(1-d_r)^2}. \tag{15}
\end{align}
Collapse receptor basis per synapse type:
\begin{align}
\beta_{\text{syn-type}} = \sum_r b_r\,\text{gain}_r. \tag{16}
\end{align}
Base no-delay effective edge weight:
\begin{align}
\omega_{ji}^{\text{base}} = B_{i,j}\,\ell_s\,w_{ji}\,\beta_{\text{syn-type}(ji)}. \tag{17}
\end{align}
Delay-aware envelope correction (from (11)):
\begin{align}
\omega_{ji}(q) = \omega_{ji}^{\text{base}}\,q^{d_{ji}-1}. \tag{18}
\end{align}
This is the implemented delay-aware correction.

### 6) Mean-field proxy for $\gamma=1$

Define effective surrogate slope after refractory gating:

\begin{align}
\bar s_{\mathrm{eff}}(r) = \eta_{\mathrm{ref}}(r)\,\bar s_{\mathrm{raw}}(r),
\quad
\eta_{\mathrm{ref}}(r)\approx 1-r\,\bar t_{\mathrm{ref}}/1000. \tag{19}
\end{align}

Then the local factor proxy is

\begin{align}
\rho_{\mathrm{loc}}(r)
\approx
\left|\bar\alpha_m + \bar s_{\mathrm{eff}}(r)(\bar\kappa_{\mathrm{asc}}-1)\right|. \tag{20}
\end{align}

Set envelope $q=\rho_{\mathrm{loc}}(r)$ (clipped to $[0,1)$), define delay factor

\begin{align}
\chi_{\mathrm{delay}}(r)=\mathbb E\left[q^{d-1}\right], \tag{21}
\end{align}

and

\begin{align}
\rho_W^{\mathrm{eff}}(r)=\rho(W_{\mathrm{base}})\,\chi_{\mathrm{delay}}(r). \tag{22}
\end{align}

Finally,

\begin{align}
\rho_{\mathrm{proxy}}(r)
\approx
\rho_{\mathrm{loc}}(r)
+ \alpha_{\mathrm{rec}}\,\bar s_{\mathrm{eff}}(r)\,\rho_W^{\mathrm{eff}}(r). \tag{23}
\end{align}

Critical value:

\begin{align}
\alpha_{\mathrm{crit}}(r;\gamma=1)
=
\frac{1-\rho_{\mathrm{loc}}(r)}{\bar s_{\mathrm{eff}}(r)\,\rho_W^{\mathrm{eff}}(r)}. \tag{24}
\end{align}
Robust recommendation over operating-rate set $\mathcal R$:
\begin{align}
\alpha_{\mathrm{rec}}^*=\kappa\min_{r\in\mathcal R}\alpha_{\mathrm{crit}}(r),
\qquad \kappa\in[0.5,0.8]. \tag{25}
\end{align}

Equations (15)–(25) are exactly what the updated calculation cell implements.